### Imports and data

In [18]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

data = load_breast_cancer()
X, y = data.data, data.target

print("Feature matrix shape:", X.shape)
print("Target classes:", np.unique(y))


Feature matrix shape: (569, 30)
Target classes: [0 1]


In [19]:
df = pd.DataFrame(X, columns=data.feature_names)
df[["mean radius", "mean area", "mean smoothness", "mean symmetry"]].describe().loc[["min", "max", "mean", "std"]]


,mean radius,mean area,mean smoothness,mean symmetry
min,6.981000,143.500000,0.052630,0.106000
max,28.110000,2501.000000,0.163400,0.304000
mean,14.127292,654.889104,0.096360,0.181162
std,3.524049,351.914129,0.014064,0.027414


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train size:", X_train.shape[0], "| Test size:", X_test.shape[0])


Train size: 455 | Test size: 114


### Part A: Without Scaling

##### We'll store each accuracy in `results_before` as we go.

In [22]:
results_before = {}


#### Model 1: Logistic Regression (Before Scaling)

In [23]:
log_reg = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
log_reg.fit(X_train, y_train)

preds = log_reg.predict(X_test)
acc = accuracy_score(y_test, preds)
results_before["Logistic Regression"] = acc

print(f"Logistic Regression accuracy (no scaling): {acc:.4f}")


Logistic Regression accuracy (no scaling): 0.9649


#### Model 2: KNN (Before Scaling)

In [24]:
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

preds = knn.predict(X_test)
acc = accuracy_score(y_test, preds)
results_before["KNN"] = acc

print(f"KNN accuracy (no scaling): {acc:.4f}")


KNN accuracy (no scaling): 0.9123


#### Model 3: Decision Tree (Before Scaling)

In [25]:
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt.fit(X_train, y_train)

preds = dt.predict(X_test)
acc = accuracy_score(y_test, preds)
results_before["Decision Tree"] = acc

print(f"Decision Tree accuracy (no scaling): {acc:.4f}")


Decision Tree accuracy (no scaling): 0.9123


#### Model 4: Random Forest (Before Scaling)

In [26]:
rf = RandomForestClassifier(random_state=RANDOM_STATE)
rf.fit(X_train, y_train)

preds = rf.predict(X_test)
acc = accuracy_score(y_test, preds)
results_before["Random Forest"] = acc

print(f"Random Forest accuracy (no scaling): {acc:.4f}")


Random Forest accuracy (no scaling): 0.9561


In [27]:
print("Summary — Before Scaling")
for name, acc in results_before.items():
    print(f"{name:20s}: {acc:.4f}")


Summary — Before Scaling
Logistic Regression : 0.9649
KNN                 : 0.9123
Decision Tree       : 0.9123
Random Forest       : 0.9561


### Part B : With `StandardScaler`

Same four models, same train/test split — but this time each one is wrapped in a
`Pipeline(StandardScaler -> model)`. The pipeline fits the scaler only on training
data and applies that same transform to the test data, so there's no data leakage.


In [28]:
results_after = {}


#### Model 1: Logistic Regression (After Scaling)

In [29]:
log_reg_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)),
])
log_reg_scaled.fit(X_train, y_train)

preds = log_reg_scaled.predict(X_test)
acc = accuracy_score(y_test, preds)
results_after["Logistic Regression"] = acc

print(f"Logistic Regression accuracy (scaled): {acc:.4f}")


Logistic Regression accuracy (scaled): 0.9825


#### Model 2: KNN (After Scaling)

In [30]:
knn_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier()),
])
knn_scaled.fit(X_train, y_train)

preds = knn_scaled.predict(X_test)
acc = accuracy_score(y_test, preds)
results_after["KNN"] = acc

print(f"KNN accuracy (scaled): {acc:.4f}")


KNN accuracy (scaled): 0.9561


#### Model 3: Decision Tree (After Scaling)

In [31]:
dt_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
])
dt_scaled.fit(X_train, y_train)

preds = dt_scaled.predict(X_test)
acc = accuracy_score(y_test, preds)
results_after["Decision Tree"] = acc

print(f"Decision Tree accuracy (scaled): {acc:.4f}")


Decision Tree accuracy (scaled): 0.9123


#### Model 4: Random Forest (After Scaling)

In [32]:
rf_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])
rf_scaled.fit(X_train, y_train)

preds = rf_scaled.predict(X_test)
acc = accuracy_score(y_test, preds)
results_after["Random Forest"] = acc

print(f"Random Forest accuracy (scaled): {acc:.4f}")


Random Forest accuracy (scaled): 0.9561


In [33]:
print("Summary — After Scaling")
for name, acc in results_after.items():
    print(f"{name:20s}: {acc:.4f}")


Summary — After Scaling
Logistic Regression : 0.9825
KNN                 : 0.9561
Decision Tree       : 0.9123
Random Forest       : 0.9561


### Comparison

In [34]:
comparison = pd.DataFrame({
    "Before Scaling": results_before,
    "After Scaling": results_after,
})
comparison["Change"] = comparison["After Scaling"] - comparison["Before Scaling"]
comparison = comparison.round(4)
comparison


,Before Scaling,After Scaling,Change
Logistic Regression,0.9649,0.9825,0.0175
KNN,0.9123,0.9561,0.0439
Decision Tree,0.9123,0.9123,0.0000
Random Forest,0.9561,0.9561,0.0000
